# Grid Arima. Again
Целью этого блокнота было улучшить результаты Arima
<br />После нескольких часов было обнаружено, что для данных корректно брать параметры с следущим условием 
(p <= 2, d <= 2, q <= 1) или (p <= 1, d <= 1)
и получены следущие результаты.
| Метрика | Значение |
| ------- | -------- |
| MAPE    | 0.020    |
| RMSE    | 0.024    |
| SDEV    | 0.013    |

Т.е. улучшили на 0.3% MAPE

In [1]:
import numpy as np
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
from itertools import product
from tqdm import tqdm
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)
df = pd.read_csv("final/just_pop.csv").sort_values(["country_code", "year"])

# Расширенный grid search параметров
p_values = [0, 1, 2, 3]
d_values = [0, 1, 2, 3]
q_values = [0, 1, 2, 3]
param_grid = list(product(p_values, d_values, q_values))

train_end = 2012
years = list(range(2013, 2023))

problem_countries = []

print("Поиск проблемных стран с расширенным grid search...")
for c in tqdm(df["country_code"].unique()):
    d = df[df["country_code"] == c].copy()

    y_train_full = d[d["year"] <= train_end]["Population_total"].values

    if len(y_train_full) < 20:  # Пропускаем страны с недостаточным количеством данных
        continue

    # Валидация для выбора лучших параметров
    val_years = max(10, int(len(y_train_full) * 0.2))
    y_train = y_train_full[:-val_years]
    y_val = y_train_full[-val_years:]

    best_order = (1, 1, 0)  # дефолтные параметры
    best_mape = float('inf')
    found_valid_order = False

    # Grid search для выбора лучших параметров
    for order in param_grid:
        try:
            model = ARIMA(y_train, order=order)
            model_fit = model.fit()
            forecast = model_fit.forecast(steps=len(y_val))
            val_mape = mape(y_val, forecast)

            if val_mape < best_mape:
                best_mape = val_mape
                best_order = order
                found_valid_order = True
        except:
            continue

    # Если не нашли валидный порядок, пропускаем
    if not found_valid_order:
        problem_countries.append(c)
        continue

    # Проверяем, можем ли мы использовать лучшие параметры для финального прогноза
    y_train = y_train_full.copy()
    is_problematic = False

    for year in years:
        try:
            model = ARIMA(y_train, order=best_order)
            model_fit = model.fit()
            forecast = model_fit.forecast(steps=1)
            y_hat = forecast[0]
            y_train = np.append(y_train, y_hat)
        except Exception as e:
            # Перехватываем все ошибки (включая LinAlgError)
            # Любая ошибка при обучении означает проблемную страну
            problem_countries.append(c)
            is_problematic = True
            break

    if is_problematic:
        continue

print(f"\nНайдено проблемных стран: {len(problem_countries)}")
print(f"Проблемные страны: {problem_countries}")


Поиск проблемных стран с расширенным grid search...


  0%|          | 0/207 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
100%|██████████| 207/207 [28:58<00:00,  8.40s/it]


Найдено проблемных стран: 207
Проблемные страны: ['AFE', 'AFG', 'AFW', 'AGO', 'ALB', 'ARB', 'ARE', 'ARG', 'ARM', 'AUS', 'AUT', 'AZE', 'BDI', 'BEL', 'BEN', 'BFA', 'BGD', 'BGR', 'BHR', 'BIH', 'BLR', 'BOL', 'BRA', 'BWA', 'CAF', 'CAN', 'CEB', 'CHE', 'CHL', 'CHN', 'CIV', 'CMR', 'COD', 'COG', 'COL', 'CRI', 'CSS', 'CUB', 'CYP', 'CZE', 'DEU', 'DJI', 'DNK', 'DOM', 'DZA', 'EAP', 'EAR', 'EAS', 'ECA', 'ECS', 'ECU', 'EGY', 'EMU', 'ERI', 'ESP', 'EST', 'ETH', 'EUU', 'FCS', 'FIN', 'FRA', 'GAB', 'GBR', 'GEO', 'GHA', 'GIN', 'GMB', 'GNB', 'GNQ', 'GRC', 'GTM', 'HIC', 'HKG', 'HND', 'HPC', 'HRV', 'HTI', 'HUN', 'IBD', 'IBT', 'IDA', 'IDB', 'IDN', 'IDX', 'IND', 'IRL', 'IRN', 'IRQ', 'ISR', 'ITA', 'JAM', 'JOR', 'JPN', 'KAZ', 'KEN', 'KGZ', 'KHM', 'KOR', 'KWT', 'LAC', 'LAO', 'LBN', 'LBR', 'LBY', 'LCN', 'LDC', 'LIC', 'LKA', 'LMC', 'LMY', 'LSO', 'LTE', 'LTU', 'LVA', 'MAR', 'MDA', 'MDG', 'MEA', 'MEX', 'MIC', 'MKD', 'MLI', 'MMR', 'MNA', 'MNG', 'MOZ', 'MRT', 'MUS', 'MWI', 'MYS', 'NAC', 'NAM', 'NER', 'NGA', 'NIC', 'NLD

In [5]:
import pandas as pd
import numpy as np

df_just_pop = pd.read_csv("final/just_pop.csv").sort_values(["country_code", "year"])
#df = pd.read_csv("output/merged_worldbank_long_no_nan.csv")
def naive_forecast(train_series, horizon):
    last_value = train_series.iloc[-1]
    return np.array([last_value] * horizon)

def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true))

def rmse_perc(y_true, y_pred):
    rmse_val = np.sqrt(np.mean((y_true - y_pred)**2))
    return rmse_val / np.mean(y_true)

def sdev_perc(y_true, y_pred):
    e = np.abs(y_true - y_pred)
    return e.std() / np.mean(y_true)

models_results = {}
def print_model_summary(model_name):
    for train_type, metrics in models_results.get(model_name, {}).items():
        print(f"\n=== Model: {model_name} | Train type: {train_type} ===")
        line = f"MAPE={metrics.get('MAPE', float('nan')):.3f}, " \
               f"RMSE={metrics.get('RMSE', float('nan')):.3f}, " \
               f"SDEV={metrics.get('SDEV', float('nan')):.3f}"
        print(line)


In [8]:
import numpy as np
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
from itertools import product
from tqdm import tqdm
models_results = {}
df = pd.read_csv("final/just_pop.csv").sort_values(["country_code", "year"])

models_results['ARIMA_grid'] = {}
res_map = {"MAPE": [], "RMSE": [], "SDEV": []}

# Grid search параметров
p_values = [0, 1, 2, 3]
d_values = [0, 1, 2]
q_values = [0, 1, 2]
param_grid = list(product(p_values, d_values, q_values))

train_end = 2012
years = list(range(2013, 2023))
successful_countries = 0
for i, c in enumerate(tqdm(df["country_code"].unique()), start=1):
    d = df[df["country_code"] == c].copy()

    y_train_full = d[d["year"] <= train_end]["Population_total"].values

    # Валидация для выбора лучших параметров
    # Используем последние 10 лет для валидации (или 20% данных, минимум 10)
    val_years = max(10, int(len(y_train_full) * 0.2))
    y_train = y_train_full[:-val_years]
    y_val = y_train_full[-val_years:]

    best_order = (1, 1, 0)  # дефолтные параметры
    best_mape = float('inf')

    # Grid search для выбора лучших параметров
    for order in param_grid:
        try:
            # Обучаем модель на тренировочных данных
            model = ARIMA(y_train, order=order)
            model_fit = model.fit()

            # Прогнозируем на валидационных данных
            forecast = model_fit.forecast(steps=len(y_val))

            # Вычисляем MAPE на валидации
            val_mape = mape(y_val, forecast)

            if val_mape < best_mape:
                best_mape = val_mape
                best_order = order
        except:
            # Если модель не сходится, пропускаем эти параметры
            continue

    # Используем лучшие параметры для финального прогноза
    # train_end и years остаются такими же как в оригинале
    y_train = y_train_full.copy()
    y_true = []
    y_pred = []

    for year in years:
            model = ARIMA(y_train, order=best_order)
            model_fit = model.fit()
            forecast = model_fit.forecast(steps=1)
            y_hat = forecast[0]
            y_pred.append(y_hat)

            row = d[d["year"] == year]
            if len(row) == 1:
                y_true.append(row["Population_total"].values[0])

            y_train = np.append(y_train, y_hat)

    if len(y_true) > 0:
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        res_map["MAPE"].append(mape(y_true, y_pred))
        res_map["RMSE"].append(rmse_perc(y_true, y_pred))
        res_map["SDEV"].append(sdev_perc(y_true, y_pred))
        successful_countries += 1
    if i % 10 == 0:
        print(f"Обработано {i} стран, успешных: {successful_countries}")
models_results['ARIMA_grid']['population'] = {
    "MAPE": float(np.mean(res_map["MAPE"])),
    "RMSE": float(np.mean(res_map["RMSE"])),
    "SDEV": float(np.mean(res_map["SDEV"])),
    "params_used": best_order
}

print_model_summary('ARIMA_grid')

  0%|          | 0/207 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
  5%|▍         | 10/207 [00:52<15:14,  4.64s/it]

Обработано 10 стран, успешных: 10


 10%|▉         | 20/207 [01:41<13:04,  4.20s/it]

Обработано 20 стран, успешных: 20


 11%|█         | 22/207 [01:57<16:26,  5.33s/it]


LinAlgError: LU decomposition error.

In [10]:
country_codes = df["country_code"].unique()
c = country_codes[22]  # 22-я страна (индексация с 0)
d = df[df["country_code"] == c].copy()

y_train_full = d[d["year"] <= train_end]["Population_total"].values
val_years = max(10, int(len(y_train_full) * 0.2))
y_train = y_train_full[:-val_years]
y_val = y_train_full[-val_years:]

best_order = (1,1,0)
best_mape = float('inf')

# grid search
for order in param_grid:
    try:
        model = ARIMA(y_train, order=order)
        model_fit = model.fit()
        forecast = model_fit.forecast(steps=len(y_val))
        val_mape = mape(y_val, forecast)
        if val_mape < best_mape:
            best_mape = val_mape
            best_order = order
    except:
        continue

# финальный прогноз
y_train = y_train_full.copy()
y_true = []
y_pred = []

try:
    for year in years:
        model = ARIMA(y_train, order=best_order)
        model_fit = model.fit()
        forecast = model_fit.forecast(steps=1)
        y_hat = forecast[0]
        y_pred.append(y_hat)

        row = d[d["year"] == year]
        if len(row) == 1:
            y_true.append(row["Population_total"].values[0])

        y_train = np.append(y_train, y_hat)
except np.linalg.LinAlgError:
    print(f"LU decomposition error на стране {c}")


LU decomposition error на стране BRA


In [18]:
bra = df[df["country_code"] == "BRA"]
print(bra)

     country_code  year  Population_total
1408          BRA  1960        72388126.0
1409          BRA  1961        74605447.0
1410          BRA  1962        76865323.0
1411          BRA  1963        79164235.0
1412          BRA  1964        81488595.0
1413          BRA  1965        83817583.0
1414          BRA  1966        86139359.0
1415          BRA  1967        88446124.0
1416          BRA  1968        90741240.0
1417          BRA  1969        93045777.0
1418          BRA  1970        95375651.0
1419          BRA  1971        97739273.0
1420          BRA  1972       100146797.0
1421          BRA  1973       102600976.0
1422          BRA  1974       105089675.0
1423          BRA  1975       107619565.0
1424          BRA  1976       110213349.0
1425          BRA  1977       112875292.0
1426          BRA  1978       115600942.0
1427          BRA  1979       118380821.0
1428          BRA  1980       121207461.0
1429          BRA  1981       124063109.0
1430          BRA  1982       1269

Будем перебирать параметры для Бразилии

In [25]:
import numpy as np
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
from itertools import product
from tqdm import tqdm

df = pd.read_csv("final/just_pop.csv").sort_values(["country_code", "year"])

models_results['ARIMA_grid'] = {}
res_map = {"MAPE": [], "RMSE": [], "SDEV": []}

# Grid search параметров
p_values = [0, 1]
d_values = [0, 1]
q_values = [0, 1, 2, 3, 4]
param_grid = list(product(p_values, d_values, q_values))

train_end = 2012
years = list(range(2013, 2023))
successful_countries = 0
for c in 'BRA':
    d = df[df["country_code"] == c].copy()

    y_train_full = d[d["year"] <= train_end]["Population_total"].values
    val_years = 10
    y_train = y_train_full[:-val_years]
    y_val = y_train_full[-val_years:]

    best_order = (1, 1, 0)  # дефолтные параметры
    best_mape = float('inf')

    # Grid search для выбора лучших параметров
    for order in param_grid:
        try:
            # Обучаем модель на тренировочных данных
            model = ARIMA(y_train, order=order)
            model_fit = model.fit()

            # Прогнозируем на валидационных данных
            forecast = model_fit.forecast(steps=len(y_val))

            # Вычисляем MAPE на валидации
            val_mape = mape(y_val, forecast)

            if val_mape < best_mape:
                best_mape = val_mape
                best_order = order
        except:
            # Если модель не сходится, пропускаем эти параметры
            continue

    # Используем лучшие параметры для финального прогноза
    # train_end и years остаются такими же как в оригинале
    y_train = y_train_full.copy()
    y_true = []
    y_pred = []

    for year in years:
            model = ARIMA(y_train, order=best_order)
            model_fit = model.fit()
            forecast = model_fit.forecast(steps=1)
            y_hat = forecast[0]
            y_pred.append(y_hat)

            row = d[d["year"] == year]
            if len(row) == 1:
                y_true.append(row["Population_total"].values[0])

            y_train = np.append(y_train, y_hat)

    if len(y_true) > 0:
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        res_map["MAPE"].append(mape(y_true, y_pred))
        res_map["RMSE"].append(rmse_perc(y_true, y_pred))
        res_map["SDEV"].append(sdev_perc(y_true, y_pred))
        successful_countries += 1
    if i % 20 == 0:
        print(f"Обработано {i} стран, успешных: {successful_countries}")
models_results['ARIMA_grid']['population'] = {
    "MAPE": float(np.mean(res_map["MAPE"])),
    "RMSE": float(np.mean(res_map["RMSE"])),
    "SDEV": float(np.mean(res_map["SDEV"])),
    "params_used": best_order
}

print(f"Successful countries count: {successful_countries}")

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:1026: RuntimeWarning: invalid value encountered in scalar divide
  params_variance = np.inner(endog, endog) / self.nobs
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:534: RuntimeWarning: invalid value encountered in scalar divide
  return -self.loglike(params, *args) / nobs
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for ARMA and trend. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:4008: RuntimeWarning: Degrees of freedom <= 0 for slice
  return _methods._var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:175: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_d

Successful countries count: 0


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [12]:
import pmdarima as pm

y_train = df[(df["country_code"]=="BRA") & (df["year"] <= 2012)]["Population_total"].values
model = pm.auto_arima(
    y_train, start_p=0, max_p=3,
    start_q=0, max_q=3,
    max_d=2, seasonal=False,
    error_action='ignore', suppress_warnings=True
)
forecast = model.predict(n_periods=10)  # например, на 2013-2022
print(forecast)


ModuleNotFoundError: No module named 'pmdarima'

Дефолтный грид

In [14]:
import numpy as np
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
from itertools import product
from tqdm import tqdm

df = pd.read_csv("final/just_pop.csv").sort_values(["country_code", "year"])

models_results['ARIMA_grid'] = {}
res_map = {"MAPE": [], "RMSE": [], "SDEV": []}

# Grid search параметров
p_values = [0, 1, 2]
d_values = [0, 1, 2]
q_values = [0, 1]
param_grid = list(product(p_values, d_values, q_values))

train_end = 2012
years = list(range(2013, 2023))
successful_countries = 0
for c in tqdm(df["country_code"].unique()):
    d = df[df["country_code"] == c].copy()

    y_train_full = d[d["year"] <= train_end]["Population_total"].values

    # Валидация для выбора лучших параметров
    # Используем последние 10 лет для валидации (или 20% данных, минимум 10)
    val_years = max(10, int(len(y_train_full) * 0.2))
    y_train = y_train_full[:-val_years]
    y_val = y_train_full[-val_years:]

    best_order = (1, 1, 0)  # дефолтные параметры
    best_mape = float('inf')

    # Grid search для выбора лучших параметров
    for order in param_grid:
        try:
            # Обучаем модель на тренировочных данных
            model = ARIMA(y_train, order=order)
            model_fit = model.fit()

            # Прогнозируем на валидационных данных
            forecast = model_fit.forecast(steps=len(y_val))

            # Вычисляем MAPE на валидации
            val_mape = mape(y_val, forecast)

            if val_mape < best_mape:
                best_mape = val_mape
                best_order = order
        except:
            # Если модель не сходится, пропускаем эти параметры
            continue

    # Используем лучшие параметры для финального прогноза
    # train_end и years остаются такими же как в оригинале
    y_train = y_train_full.copy()
    y_true = []
    y_pred = []

    for year in years:
            model = ARIMA(y_train, order=best_order)
            model_fit = model.fit()
            forecast = model_fit.forecast(steps=1)
            y_hat = forecast[0]
            y_pred.append(y_hat)

            row = d[d["year"] == year]
            if len(row) == 1:
                y_true.append(row["Population_total"].values[0])

            y_train = np.append(y_train, y_hat)

    if len(y_true) > 0:
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        res_map["MAPE"].append(mape(y_true, y_pred))
        res_map["RMSE"].append(rmse_perc(y_true, y_pred))
        res_map["SDEV"].append(sdev_perc(y_true, y_pred))
        successful_countries += 1
    if i % 20 == 0:
        print(f"Обработано {i} стран, успешных: {successful_countries}")
models_results['ARIMA_grid']['population'] = {
    "MAPE": float(np.mean(res_map["MAPE"])),
    "RMSE": float(np.mean(res_map["RMSE"])),
    "SDEV": float(np.mean(res_map["SDEV"])),
    "params_used": best_order
}

print_model_summary('ARIMA_grid')

  0%|          | 0/207 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
100%|██████████| 207/207 [06:30<00:00,  1.89s/it]


=== Model: ARIMA_grid | Train type: population ===
MAPE=0.023, RMSE=0.028, SDEV=0.015


Проверим на 30 странах что Successful

In [16]:
import numpy as np
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
from itertools import product
from tqdm import tqdm

df = pd.read_csv("final/just_pop.csv").sort_values(["country_code", "year"])

models_results['ARIMA_grid'] = {}
res_map = {"MAPE": [], "RMSE": [], "SDEV": []}

# Grid search параметров
p_values = [0, 1, 2]
d_values = [0, 1, 2]
q_values = [0, 1]
param_grid = list(product(p_values, d_values, q_values))

train_end = 2012
years = list(range(2013, 2023))
successful_countries = 0
for c in tqdm(df["country_code"].unique()[:30]):
    d = df[df["country_code"] == c].copy()

    y_train_full = d[d["year"] <= train_end]["Population_total"].values
    val_years = 10
    y_train = y_train_full[:-val_years]
    y_val = y_train_full[-val_years:]

    best_order = (1, 1, 0)  # дефолтные параметры
    best_mape = float('inf')

    # Grid search для выбора лучших параметров
    for order in param_grid:
        try:
            # Обучаем модель на тренировочных данных
            model = ARIMA(y_train, order=order)
            model_fit = model.fit()

            # Прогнозируем на валидационных данных
            forecast = model_fit.forecast(steps=len(y_val))

            # Вычисляем MAPE на валидации
            val_mape = mape(y_val, forecast)

            if val_mape < best_mape:
                best_mape = val_mape
                best_order = order
        except:
            # Если модель не сходится, пропускаем эти параметры
            continue

    # Используем лучшие параметры для финального прогноза
    # train_end и years остаются такими же как в оригинале
    y_train = y_train_full.copy()
    y_true = []
    y_pred = []

    for year in years:
            model = ARIMA(y_train, order=best_order)
            model_fit = model.fit()
            forecast = model_fit.forecast(steps=1)
            y_hat = forecast[0]
            y_pred.append(y_hat)

            row = d[d["year"] == year]
            if len(row) == 1:
                y_true.append(row["Population_total"].values[0])

            y_train = np.append(y_train, y_hat)

    if len(y_true) > 0:
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        res_map["MAPE"].append(mape(y_true, y_pred))
        res_map["RMSE"].append(rmse_perc(y_true, y_pred))
        res_map["SDEV"].append(sdev_perc(y_true, y_pred))
        successful_countries += 1
    if i % 20 == 0:
        print(f"Обработано {i} стран, успешных: {successful_countries}")
models_results['ARIMA_grid']['population'] = {
    "MAPE": float(np.mean(res_map["MAPE"])),
    "RMSE": float(np.mean(res_map["RMSE"])),
    "SDEV": float(np.mean(res_map["SDEV"])),
    "params_used": best_order
}

print(f"Successful countries count: {successful_countries}")

  0%|          | 0/30 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
100%|██████████| 30/30 [01:11<00:00,  2.40s/it]

Successful countries count: 30


In [24]:
import numpy as np
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
from itertools import product
from tqdm import tqdm

df = pd.read_csv("final/just_pop.csv").sort_values(["country_code", "year"])

models_results['ARIMA_grid'] = {}
res_map = {"MAPE": [], "RMSE": [], "SDEV": []}

# Grid search параметров
p_values = [0, 1, 2]
d_values = [0, 1, 2]# порядок дифференциирования
q_values = [0, 1, 2]
param_grid = list(product(p_values, d_values, q_values))

train_end = 2012
years = list(range(2013, 2023))
successful_countries = 0
for c in tqdm(df["country_code"].unique()):
    d = df[df["country_code"] == c].copy()

    y_train_full = d[d["year"] <= train_end]["Population_total"].values
    val_years = 10
    y_train = y_train_full[:-val_years]
    y_val = y_train_full[-val_years:]

    best_order = (1, 1, 0)  # дефолтные параметры
    best_mape = float('inf')

    # Grid search для выбора лучших параметров
    for order in param_grid:
        try:
            # Обучаем модель на тренировочных данных
            model = ARIMA(y_train, order=order)
            model_fit = model.fit()

            # Прогнозируем на валидационных данных
            forecast = model_fit.forecast(steps=len(y_val))

            # Вычисляем MAPE на валидации
            val_mape = mape(y_val, forecast)

            if val_mape < best_mape:
                best_mape = val_mape
                best_order = order
        except:
            # Если модель не сходится, пропускаем эти параметры
            continue

    # Используем лучшие параметры для финального прогноза
    # train_end и years остаются такими же как в оригинале
    y_train = y_train_full.copy()
    y_true = []
    y_pred = []

    for year in years:
            model = ARIMA(y_train, order=best_order)
            model_fit = model.fit()
            forecast = model_fit.forecast(steps=1)
            y_hat = forecast[0]
            y_pred.append(y_hat)

            row = d[d["year"] == year]
            if len(row) == 1:
                y_true.append(row["Population_total"].values[0])

            y_train = np.append(y_train, y_hat)

    if len(y_true) > 0:
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        res_map["MAPE"].append(mape(y_true, y_pred))
        res_map["RMSE"].append(rmse_perc(y_true, y_pred))
        res_map["SDEV"].append(sdev_perc(y_true, y_pred))
        successful_countries += 1
    if i % 20 == 0:
        print(f"Обработано {i} стран, успешных: {successful_countries}")
models_results['ARIMA_grid']['population'] = {
    "MAPE": float(np.mean(res_map["MAPE"])),
    "RMSE": float(np.mean(res_map["RMSE"])),
    "SDEV": float(np.mean(res_map["SDEV"])),
    "params_used": best_order
}

print(f"Successful countries count: {successful_countries}")
print_model_summary("ARIMA_grid")

  0%|          | 0/207 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
100%|██████████| 207/207 [12:12<00:00,  3.54s/it]

Successful countries count: 207


In [26]:
print_model_summary("ARIMA_grid")


=== Model: ARIMA_grid | Train type: population ===
MAPE=nan, RMSE=nan, SDEV=nan


In [28]:
import numpy as np
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
from itertools import product

def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true))

def rmse_perc(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred)**2)) / np.mean(y_true)

def sdev_perc(y_true, y_pred):
    e = np.abs(y_true - y_pred)
    return e.std() / np.mean(y_true)

df = pd.read_csv("final/just_pop.csv").sort_values(["country_code", "year"])

p_values = [0, 1, 2]
d_values = [0, 1, 2]
q_values = [0, 1, 2, 3, 4, 5]

all_combinations = list(product(p_values, d_values, q_values))

# (p <= 2, d <= 2, q <= 1) или (p <= 1, d <= 1, q <= 5)
param_grid = [ (p,d,q) for (p,d,q) in all_combinations
               if (p <= 2 and d <= 2 and q <= 1) or (p <= 1 and d <= 1 and q <= 5) ]

train_end = 2012
years = list(range(2013, 2023))

# Для одной страны
c = 'BRA'
d = df[df["country_code"] == c].copy()
y_train_full = d[d["year"] <= train_end]["Population_total"].values
val_years = 10
y_train = y_train_full[:-val_years]
y_val = y_train_full[-val_years:]

best_order = (1, 1, 0)
best_mape = float('inf')

# Grid search
for order in param_grid:
    try:
        model = ARIMA(y_train, order=order)
        model_fit = model.fit()
        forecast = model_fit.forecast(steps=len(y_val))
        val_mape = mape(y_val, forecast)
        if val_mape < best_mape:
            best_mape = val_mape
            best_order = order
    except:
        continue

# Финальный прогноз
y_train = y_train_full.copy()
y_pred = []

for year in years:
    model = ARIMA(y_train, order=best_order)
    model_fit = model.fit()
    forecast = model_fit.forecast(steps=1)
    y_hat = forecast[0]
    y_pred.append(y_hat)
    y_train = np.append(y_train, y_hat)

print(f"Лучшие параметры ARIMA для BRA: {best_order}")
print("Прогноз численности населения Бразилии на 2013-2022 годы:")
for year, pop in zip(years, y_pred):
    print(f"{year}: {int(pop):,}")


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'


Лучшие параметры ARIMA для BRA: (2, 0, 1)
Прогноз численности населения Бразилии на 2013-2022 годы:
2013: 198,449,199
2014: 199,967,693
2015: 201,431,857
2016: 202,841,880
2017: 204,198,192
2018: 205,501,042
2019: 206,750,919
2020: 207,948,198
2021: 209,093,423
2022: 210,187,073


In [29]:
import numpy as np
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
from itertools import product
from tqdm import tqdm

df = pd.read_csv("final/just_pop.csv").sort_values(["country_code", "year"])

models_results['ARIMA_grid'] = {}
res_map = {"MAPE": [], "RMSE": [], "SDEV": []}

# Grid search параметров
p_values = [0, 1, 2]
d_values = [0, 1, 2]
q_values = [0, 1, 2, 3, 4, 5]

all_combinations = list(product(p_values, d_values, q_values))

# (p <= 2, d <= 2, q <= 1) или (p <= 1, d <= 1, q <= 5)
param_grid = [ (p,d,q) for (p,d,q) in all_combinations
               if (p <= 2 and d <= 2 and q <= 1) or (p <= 1 and d <= 1 and q <= 5) ]

train_end = 2012
years = list(range(2013, 2023))
successful_countries = 0
for c in tqdm(df["country_code"].unique()[:30]):
    d = df[df["country_code"] == c].copy()

    y_train_full = d[d["year"] <= train_end]["Population_total"].values
    val_years = 10
    y_train = y_train_full[:-val_years]
    y_val = y_train_full[-val_years:]

    best_order = (1, 1, 0)  # дефолтные параметры
    best_mape = float('inf')

    # Grid search для выбора лучших параметров
    for order in param_grid:
        try:
            # Обучаем модель на тренировочных данных
            model = ARIMA(y_train, order=order)
            model_fit = model.fit()

            # Прогнозируем на валидационных данных
            forecast = model_fit.forecast(steps=len(y_val))

            # Вычисляем MAPE на валидации
            val_mape = mape(y_val, forecast)

            if val_mape < best_mape:
                best_mape = val_mape
                best_order = order
        except:
            # Если модель не сходится, пропускаем эти параметры
            continue

    # Используем лучшие параметры для финального прогноза
    # train_end и years остаются такими же как в оригинале
    y_train = y_train_full.copy()
    y_true = []
    y_pred = []

    for year in years:
            model = ARIMA(y_train, order=best_order)
            model_fit = model.fit()
            forecast = model_fit.forecast(steps=1)
            y_hat = forecast[0]
            y_pred.append(y_hat)

            row = d[d["year"] == year]
            if len(row) == 1:
                y_true.append(row["Population_total"].values[0])

            y_train = np.append(y_train, y_hat)

    if len(y_true) > 0:
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        res_map["MAPE"].append(mape(y_true, y_pred))
        res_map["RMSE"].append(rmse_perc(y_true, y_pred))
        res_map["SDEV"].append(sdev_perc(y_true, y_pred))
        successful_countries += 1
    if i % 20 == 0:
        print(f"Обработано {i} стран, успешных: {successful_countries}")
models_results['ARIMA_grid']['population'] = {
    "MAPE": float(np.mean(res_map["MAPE"])),
    "RMSE": float(np.mean(res_map["RMSE"])),
    "SDEV": float(np.mean(res_map["SDEV"])),
    "params_used": best_order
}

print(f"Successful countries count: {successful_countries}")
print_model_summary("ARIMA_grid")

  0%|          | 0/30 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
100%|██████████| 30/30 [01:56<00:00,  3.88s/it]

Successful countries count: 30

=== Model: ARIMA_grid | Train type: population ===
MAPE=0.022, RMSE=0.026, SDEV=0.013


In [7]:
import numpy as np
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
from itertools import product
from tqdm import tqdm

df = pd.read_csv("final/just_pop.csv").sort_values(["country_code", "year"])
models_results['ARIMA_grid'] = {}
res_map = {"MAPE": [], "RMSE": [], "SDEV": []}

# Grid search параметров
p_values = [0, 1, 2]
d_values = [0, 1, 2]
q_values = [0, 1, 2, 3, 4, 5, 7]

all_combinations = list(product(p_values, d_values, q_values))

# (p <= 2, d <= 2, q <= 1) или (p <= 1, d <= 1, q <= 5)
param_grid = [ (p,d,q) for (p,d,q) in all_combinations
               if (p <= 2 and d <= 2 and q <= 1) or (p <= 1 and d <= 1) ]

train_end = 2012
years = list(range(2013, 2023))
successful_countries = 0
for c in tqdm(df["country_code"].unique()):
    d = df[df["country_code"] == c].copy()

    y_train_full = d[d["year"] <= train_end]["Population_total"].values
    val_years = 10
    y_train = y_train_full[:-val_years]
    y_val = y_train_full[-val_years:]

    best_order = (1, 1, 0)  # дефолтные параметры
    best_mape = float('inf')

    # Grid search для выбора лучших параметров
    for order in param_grid:
        try:
            # Обучаем модель на тренировочных данных
            model = ARIMA(y_train, order=order)
            model_fit = model.fit()

            # Прогнозируем на валидационных данных
            forecast = model_fit.forecast(steps=len(y_val))

            # Вычисляем MAPE на валидации
            val_mape = mape(y_val, forecast)

            if val_mape < best_mape:
                best_mape = val_mape
                best_order = order
        except:
            # Если модель не сходится, пропускаем эти параметры
            continue

    # Используем лучшие параметры для финального прогноза
    # train_end и years остаются такими же как в оригинале
    y_train = y_train_full.copy()
    y_true = []
    y_pred = []

    for year in years:
            model = ARIMA(y_train, order=best_order)
            model_fit = model.fit()
            forecast = model_fit.forecast(steps=1)
            y_hat = forecast[0]
            y_pred.append(y_hat)

            row = d[d["year"] == year]
            if len(row) == 1:
                y_true.append(row["Population_total"].values[0])

            y_train = np.append(y_train, y_hat)

    if len(y_true) > 0:
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        res_map["MAPE"].append(mape(y_true, y_pred))
        res_map["RMSE"].append(rmse_perc(y_true, y_pred))
        res_map["SDEV"].append(sdev_perc(y_true, y_pred))
        successful_countries += 1
models_results['ARIMA_grid']['population'] = {
    "MAPE": float(np.mean(res_map["MAPE"])),
    "RMSE": float(np.mean(res_map["RMSE"])),
    "SDEV": float(np.mean(res_map["SDEV"])),
    "params_used": best_order
}

print(f"Successful countries count: {successful_countries}")
print_model_summary("ARIMA_grid")

  0%|          | 0/207 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-package

Successful countries count: 207

=== Model: ARIMA_grid | Train type: population ===
MAPE=0.020, RMSE=0.024, SDEV=0.013
